<a href="https://colab.research.google.com/github/izabelapastuszka/python-data-analysis/blob/main/language_learning_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sqlite3

import os

os.listdir('/content/drive/MyDrive/Nauka języków projekt')

['language-learning-sample-database-v2-csv-user_progress.gsheet',
 'language-learning-sample-database-v2-csv-users.gsheet',
 'language_learning.db']

In [ ]:
users_url = "https://docs.google.com/spreadsheets/d/1bde03rEdjlwnM7A5wS5nuDIx9TxjcX0VKIYgAyB6ve0/export?format=csv&gid=1384241965"

progress_url = "https://docs.google.com/spreadsheets/d/1ssZktFRDsU1fkSSLAvETUKl2zHF24MOvh3QQozvTJlc/export?format=csv&gid=2028130171"

users = pd.read_csv(users_url)
user_progress = pd.read_csv(progress_url)

print("Tabela users:", users.shape)
print("Tabela user_progress:", user_progress.shape)

Tabela users: (125, 6)
Tabela user_progress: (1000, 6)


In [ ]:
conn = sqlite3.connect("language_learning.db")

users.to_sql("users", conn, if_exists="replace", index=False)
user_progress.to_sql("user_progress", conn, if_exists="replace", index=False)

print("Tabele zapisane w bazie SQLite.")

Tabele zapisane w bazie SQLite.


In [ ]:
db_path = '/content/drive/MyDrive/Nauka języków projekt/language_learning.db'

backup_conn = sqlite3.connect(db_path)

users.to_sql('users', backup_conn, if_exists='replace', index=False)
user_progress.to_sql('user_progress', backup_conn, if_exists='replace', index=False)

backup_conn.close()

print("Baza zapisana na Dysku Google!")

Baza zapisana na Dysku Google!


In [ ]:
db_path = '/content/drive/MyDrive/Nauka języków projekt/language_learning.db'
conn = sqlite3.connect(db_path)

In [ ]:
query = """
SELECT
    COUNT(*) AS liczba_wierszy,
    COUNT(DISTINCT user_id) AS unikalne_id
FROM users;
"""

pd.read_sql_query(query, conn)

,liczba_wierszy,unikalne_id
0,125,125


In [ ]:

query = """
SELECT COUNT(*) AS puste_id
FROM users
WHERE user_id IS NULL;
"""

pd.read_sql_query(query, conn)

,puste_id
0,0


In [ ]:
query = """
SELECT COUNT(*) AS puste_id
FROM user_progress
WHERE user_id IS NULL;
"""

pd.read_sql_query(query, conn)

,puste_id
0,0


In [ ]:
query = """
SELECT COUNT(*) AS niedopasowane
FROM user_progress
LEFT JOIN users
ON user_progress.user_id = users.user_id
WHERE users.user_id IS NULL;
"""

pd.read_sql_query(query, conn)

,niedopasowane
0,0


In [ ]:
users.head(5)

,user_id,date_joined,email,password_hash,user_type,username
0,1,2022-12-08 03:05:13,aliya_oreilly8@yahoo.com,d9adc7c98f0e3b662597f9729032aca9,learner,Teagan.Schowalter49
1,2,2022-09-13 21:25:41,abbigail.ritchie62@gmail.com,6cd067e18858ccd3bebb49f9486298d4,learner,Stan.Schumm40
2,3,2024-06-17 03:14:49,odell.reinger-senger@gmail.com,4e86a7843ad4b36300d64af85614f4fe,learner,Leroy_Predovic55
3,4,2023-09-07 15:12:25,wilbur_altenwerth-kassulke4@gmail.com,41ddd91687420de888ec1a442a0b34a0,learner,Elvera_Block
4,5,2025-05-14 03:31:23,randi.sauer@gmail.com,fc5efe391ec4d7a0cde672bf0ee6caca,learner,Marvin.Effertz


In [ ]:
user_progress.head(3)


,progress_id,course_id,last_accessed,lesson_id,progress_percentage,user_id
0,1,14,2024-04-11 07:10:11,433,0.6,37
1,2,41,2022-05-25 20:21:39,543,88.3,66
2,3,39,2022-05-29 07:40:15,27,3.4,8


In [ ]:
query = """
SELECT COUNT(user_id), user_type
FROM users
GROUP BY user_type;
"""

pd.read_sql_query(query, conn)

,COUNT(user_id),user_type
0,113,learner
1,12,tutor


In [ ]:
query = """
SELECT
  ((1.0 * (SELECT COUNT(user_type) FROM users WHERE user_type='learner') /
  (SELECT COUNT(*) FROM users)) * 100) AS procent_learners;
"""

pd.read_sql_query(query, conn)

,procent_learners
0,90.4


In [ ]:
query = """
SELECT
    user_type,
    COUNT(*) AS liczba_osob,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM users), 1) AS procent
FROM users
GROUP BY user_type;
"""

pd.read_sql_query(query, conn)

,user_type,liczba_osob,procent
0,learner,113,90.4
1,tutor,12,9.6


In [ ]:
query = """
SELECT users.user_id, users.username, user_progress.progress_percentage
FROM user_progress
INNER JOIN users
ON user_progress.user_id = users.user_id;
"""

pd.read_sql_query(query, conn)

,user_id,username,progress_percentage
0,37,Donnell_Marks63,0.6
1,66,Melany.Mosciski43,88.3
2,8,Kristian.Crist76,3.4
3,2,Stan.Schumm40,25.1
4,24,Natalie.Kunze29,7.5
...,...,...,...
995,94,Geovanny_Wolf,35.0
996,89,August.Dibbert,30.6
997,15,Shane_Langosh31,4.2
998,38,Kylee.Hermann,78.3


In [46]:
query = """
SELECT users.user_id, users.username, ROUND(AVG(user_progress.progress_percentage), 2) AS średni_postęp
FROM user_progress
INNER JOIN users
ON user_progress.user_id = users.user_id
GROUP BY users.user_id;
"""

pd.read_sql_query(query, conn)

,user_id,username,średni_postęp
0,1,Teagan.Schowalter49,32.18
1,2,Stan.Schumm40,34.27
2,3,Leroy_Predovic55,24.04
3,4,Elvera_Block,39.22
4,5,Marvin.Effertz,39.64
...,...,...,...
120,121,Sofia_Roberts,35.34
121,122,Kent.Farrell42,30.58
122,123,Roma.Leffler77,30.55
123,124,Elisha_Rice,51.58


In [48]:
query = """
SELECT
    users.user_type,
    COUNT(DISTINCT users.user_id) AS liczba_uzytkownikow,
    ROUND(AVG(user_progress.progress_percentage), 2) AS sredni_postep
FROM user_progress
INNER JOIN users
    ON user_progress.user_id = users.user_id
GROUP BY users.user_type;
"""

pd.read_sql_query(query, conn)

,user_type,liczba_uzytkownikow,sredni_postep
0,learner,113,34.02
1,tutor,12,30.28
